In [ ]:
"""Calculate future evapotranspiration changes relative to historical CWatM.

The input directory must contain:
  * CWatM_historical_monthly_ET_1995_2020.csv
  * boxplot_values_ET_<scenario>_<period>.csv

Each future file contains the historical CWatM series plus five GCM series.
The script calculates Mean, Max, Min, and standard deviation over all monthly
values in each 25-year period. Percentage change is:

    100 * (future_statistic - baseline_statistic) / baseline_statistic

For the ensemble result, changes are first calculated separately for each GCM
and are then averaged across the five GCMs, matching the ensemble logic of the
reference article. Results are written as CSV files to ``et_change_tables``.
"""

from __future__ import annotations

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd


BASELINE_FILE = "CWatM_historical_monthly_ET_1995_2020.csv"
BASELINE_START = "1995-01-01"
BASELINE_END = "2019-12-31"  # The source name says 1995-2020, but data end in 2019.
EXPECTED_MODELS = ("GFDL", "IPSL", "MPI", "MRI", "UKESM")
EXPECTED_PERIODS = {
    "near_future": ("2025-01-01", "2049-12-31", "2025-2049"),
    "mid_century": ("2050-01-01", "2074-12-31", "2050-2074"),
    "far_future": ("2075-01-01", "2099-12-31", "2075-2099"),
}
SCENARIO_NAMES = {
    "126": "Optimistic / SSP126",
    "370": "Medium / SSP370",
    "585": "Pessimistic / SSP585",
}
STATISTICS = ("Mean", "Max", "Min", "Std")
MONTH_NAMES = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December",
}
SEASON_MONTHS = {
    "Winter": (12, 1, 2),
    "Spring": (3, 4, 5),
    "Summer": (6, 7, 8),
    "Autumn": (9, 10, 11),
}
SCALE_ORDER = ("Monthly", "Seasonal", "Annual")
FILE_PATTERN = re.compile(
    r"^boxplot_values_ET_(?P<scenario>126|370|585)_"
    r"(?P<period>near_future|mid_century|far_future)\.csv$"
)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--input-dir",
        type=Path,
        default=Path(__file__).resolve().parent,
        help="Directory containing the ten source CSV files.",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=None,
        help="Output directory (default: INPUT_DIR/et_change_tables).",
    )
    parser.add_argument(
        "--std-ddof",
        type=int,
        choices=(0, 1),
        default=1,
        help="Standard-deviation convention: 1=sample (default), 0=population.",
    )
    parser.add_argument(
        "--decimals", type=int, default=2, help="Decimal places in CSV outputs."
    )
    return parser.parse_args()


def read_two_column_baseline(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    required = {"time", "CWatM"}
    if set(df.columns) != required:
        raise ValueError(f"{path.name}: expected columns {sorted(required)}, got {df.columns.tolist()}")
    df = df.rename(columns={"CWatM": "ET"})
    df["time"] = pd.to_datetime(df["time"], errors="raise")
    df["ET"] = pd.to_numeric(df["ET"], errors="raise")
    return df.sort_values("time").reset_index(drop=True)


def read_future_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    required = {
        "time", "ET", "series", "scenario", "scenario_label",
        "future_period", "future_period_label",
    }
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    df["time"] = pd.to_datetime(df["time"], errors="raise")
    df["ET"] = pd.to_numeric(df["ET"], errors="raise")
    df["scenario"] = df["scenario"].astype(str)
    if df["ET"].isna().any():
        raise ValueError(f"{path.name}: ET contains missing values")
    return df


def validate_monthly_series(
    df: pd.DataFrame, start: str, end: str, label: str
) -> None:
    expected = pd.date_range(start, end, freq="MS")
    actual = pd.DatetimeIndex(df["time"].sort_values())
    if actual.duplicated().any():
        raise ValueError(f"{label}: duplicate months found")
    if not actual.equals(expected):
        missing = expected.difference(actual)
        extra = actual.difference(expected)
        raise ValueError(
            f"{label}: monthly coverage mismatch; "
            f"missing={missing.strftime('%Y-%m').tolist()}, "
            f"extra={extra.strftime('%Y-%m').tolist()}"
        )


def describe(values: pd.Series, ddof: int) -> dict[str, float]:
    return {
        "Mean": float(values.mean()),
        "Max": float(values.max()),
        "Min": float(values.min()),
        "Std": float(values.std(ddof=ddof)),
    }


def percent_change(future: float, baseline: float) -> float:
    if np.isclose(baseline, 0.0):
        return np.nan
    return 100.0 * (future - baseline) / baseline


def make_article_style_table(
    ensemble: pd.DataFrame, period: str
) -> pd.DataFrame:
    """One-row layout analogous to the grouped scenario columns in Table 7."""
    subset = ensemble.loc[ensemble["period"] == period].copy()
    values: dict[tuple[str, str], float] = {}
    for row in subset.itertuples(index=False):
        scenario_header = f"{row.scenario_label} ({row.scenario})"
        values[(scenario_header, row.statistic + "%")] = row.change_percent
    ordered_columns = []
    for scenario in ("126", "370", "585"):
        header = f"{SCENARIO_NAMES[scenario]} ({scenario})"
        ordered_columns.extend((header, stat + "%") for stat in STATISTICS)
    table = pd.DataFrame([values], index=[EXPECTED_PERIODS[period][2]])
    table = table.reindex(columns=pd.MultiIndex.from_tuples(ordered_columns))
    table.index.name = "Future period"
    return table


def grouped_statistics(
    df: pd.DataFrame, grouping: str, ddof: int
) -> dict[str, dict[str, float]]:
    """Statistics for each calendar month or climatological season.

    Monthly statistics use the 25 occurrences of that calendar month. Seasonal
    statistics use all 75 monthly observations belonging to that season. Winter
    is December-February; since statistics are distributional, assigning
    December to the following season-year is not required here.
    """
    month = df["time"].dt.month
    if grouping == "month":
        return {
            name: describe(df.loc[month == number, "ET"], ddof)
            for number, name in MONTH_NAMES.items()
        }
    if grouping == "season":
        return {
            season: describe(df.loc[month.isin(months), "ET"], ddof)
            for season, months in SEASON_MONTHS.items()
        }
    raise ValueError(f"Unsupported grouping: {grouping}")


def multiscale_series(df: pd.DataFrame) -> dict[str, pd.Series]:
    """Build monthly, seasonal-mean, and annual-mean time series.

    Each calendar year contributes four seasonal means and one annual mean.
    Winter is the mean of January, February, and December within that calendar
    year. This convention retains exactly four complete seasons for every year
    in the supplied January-December periods.
    """
    work = df[["time", "ET"]].copy()
    work["year"] = work["time"].dt.year
    month_to_season = {
        month: season for season, months in SEASON_MONTHS.items() for month in months
    }
    work["season"] = work["time"].dt.month.map(month_to_season)

    seasonal = work.groupby(["year", "season"], sort=True)["ET"].mean()
    annual = work.groupby("year", sort=True)["ET"].mean()
    expected_years = work["year"].nunique()
    if len(seasonal) != expected_years * 4 or len(annual) != expected_years:
        raise ValueError("Cannot construct complete seasonal/annual mean series")
    return {
        "Monthly": work["ET"].reset_index(drop=True),
        "Seasonal": seasonal.reset_index(drop=True),
        "Annual": annual.reset_index(drop=True),
    }


def make_multiscale_article_table(
    ensemble: pd.DataFrame, period: str
) -> pd.DataFrame:
    subset = ensemble.loc[ensemble["period"] == period]
    columns = []
    for scenario in ("126", "370", "585"):
        header = f"{SCENARIO_NAMES[scenario]} ({scenario})"
        columns.extend((header, stat + "%") for stat in STATISTICS)
    result = pd.DataFrame(
        index=pd.Index(SCALE_ORDER, name="Time scale"),
        columns=pd.MultiIndex.from_tuples(columns),
        dtype=float,
    )
    for row in subset.itertuples(index=False):
        header = f"{row.scenario_label} ({row.scenario})"
        result.loc[row.time_scale, (header, row.statistic + "%")] = row.change_percent
    return result


def make_grouped_article_table(
    ensemble: pd.DataFrame, period: str, grouping: str
) -> pd.DataFrame:
    subset = ensemble.loc[
        (ensemble["period"] == period) & (ensemble["grouping"] == grouping)
    ]
    if grouping == "month":
        row_order = list(MONTH_NAMES.values())
        index_name = "Month"
    elif grouping == "season":
        row_order = list(SEASON_MONTHS)
        index_name = "Season"
    else:
        raise ValueError(grouping)

    columns = []
    for scenario in ("126", "370", "585"):
        header = f"{SCENARIO_NAMES[scenario]} ({scenario})"
        columns.extend((header, stat + "%") for stat in STATISTICS)
    result = pd.DataFrame(
        index=pd.Index(row_order, name=index_name),
        columns=pd.MultiIndex.from_tuples(columns),
        dtype=float,
    )
    for row in subset.itertuples(index=False):
        header = f"{row.scenario_label} ({row.scenario})"
        result.loc[row.group, (header, row.statistic + "%")] = row.change_percent
    return result


def main() -> None:
    args = parse_args()
    input_dir = args.input_dir.resolve()
    output_dir = (args.output_dir or input_dir / "et_change_tables").resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    baseline_path = input_dir / BASELINE_FILE
    if not baseline_path.exists():
        raise FileNotFoundError(baseline_path)
    baseline = read_two_column_baseline(baseline_path)
    validate_monthly_series(baseline, BASELINE_START, BASELINE_END, "CWatM baseline")
    baseline_stats = describe(baseline["ET"], args.std_ddof)

    source_files: dict[tuple[str, str], Path] = {}
    for path in input_dir.glob("boxplot_values_ET_*.csv"):
        match = FILE_PATTERN.match(path.name)
        if match:
            source_files[(match["scenario"], match["period"])] = path
    expected_keys = {(s, p) for s in SCENARIO_NAMES for p in EXPECTED_PERIODS}
    missing_keys = expected_keys.difference(source_files)
    if missing_keys:
        raise FileNotFoundError(f"Missing scenario-period files: {sorted(missing_keys)}")

    absolute_rows: list[dict[str, object]] = []
    change_rows: list[dict[str, object]] = []
    grouped_change_rows: list[dict[str, object]] = []
    multiscale_change_rows: list[dict[str, object]] = []
    baseline_grouped = {
        grouping: grouped_statistics(baseline, grouping, args.std_ddof)
        for grouping in ("season", "month")
    }
    baseline_multiscale = {
        scale: describe(series, args.std_ddof)
        for scale, series in multiscale_series(baseline).items()
    }

    for scenario, period in sorted(expected_keys):
        path = source_files[(scenario, period)]
        df = read_future_file(path)
        start, end, period_years = EXPECTED_PERIODS[period]

        file_scenarios = set(df["scenario"])
        file_periods = set(df["future_period"])
        if file_scenarios != {scenario} or file_periods != {period}:
            raise ValueError(
                f"{path.name}: internal scenario/period labels do not match filename"
            )

        historical_rows = df[df["series"].str.startswith("CWatM")]
        embedded = historical_rows[["time", "ET"]].sort_values("time").reset_index(drop=True)
        if len(embedded) != len(baseline) or not np.allclose(embedded["ET"], baseline["ET"]):
            raise ValueError(f"{path.name}: embedded CWatM baseline differs from {BASELINE_FILE}")

        models = tuple(sorted(set(df["series"]) - set(historical_rows["series"])))
        if set(models) != set(EXPECTED_MODELS):
            raise ValueError(f"{path.name}: expected models {EXPECTED_MODELS}, got {models}")

        for model in EXPECTED_MODELS:
            model_df = df.loc[df["series"] == model, ["time", "ET"]]
            validate_monthly_series(model_df, start, end, f"{path.name} / {model}")
            future_stats = describe(model_df["ET"], args.std_ddof)
            for statistic in STATISTICS:
                base_value = baseline_stats[statistic]
                future_value = future_stats[statistic]
                common = {
                    "scenario": scenario,
                    "scenario_label": SCENARIO_NAMES[scenario],
                    "period": period,
                    "period_years": period_years,
                    "model": model,
                    "statistic": statistic,
                    "baseline_value": base_value,
                    "future_value": future_value,
                }
                absolute_rows.append(common)
                change_rows.append({**common, "change_percent": percent_change(future_value, base_value)})

            for grouping in ("season", "month"):
                future_grouped = grouped_statistics(model_df, grouping, args.std_ddof)
                for group, stats in future_grouped.items():
                    for statistic in STATISTICS:
                        base_value = baseline_grouped[grouping][group][statistic]
                        future_value = stats[statistic]
                        grouped_change_rows.append(
                            {
                                "scenario": scenario,
                                "scenario_label": SCENARIO_NAMES[scenario],
                                "period": period,
                                "period_years": period_years,
                                "model": model,
                                "grouping": grouping,
                                "group": group,
                                "statistic": statistic,
                                "baseline_value": base_value,
                                "future_value": future_value,
                                "change_percent": percent_change(future_value, base_value),
                            }
                        )

            for time_scale, series in multiscale_series(model_df).items():
                future_scale_stats = describe(series, args.std_ddof)
                for statistic in STATISTICS:
                    base_value = baseline_multiscale[time_scale][statistic]
                    future_value = future_scale_stats[statistic]
                    multiscale_change_rows.append(
                        {
                            "scenario": scenario,
                            "scenario_label": SCENARIO_NAMES[scenario],
                            "period": period,
                            "period_years": period_years,
                            "model": model,
                            "time_scale": time_scale,
                            "statistic": statistic,
                            "baseline_value": base_value,
                            "future_value": future_value,
                            "change_percent": percent_change(future_value, base_value),
                        }
                    )

    absolute = pd.DataFrame(absolute_rows)
    by_model = pd.DataFrame(change_rows)
    ensemble = (
        by_model.groupby(
            ["scenario", "scenario_label", "period", "period_years", "statistic"],
            as_index=False,
            sort=False,
        )
        .agg(
            baseline_value=("baseline_value", "first"),
            ensemble_future_value=("future_value", "mean"),
            change_percent=("change_percent", "mean"),
            inter_model_sd_change_percent=("change_percent", "std"),
            model_count=("model", "nunique"),
        )
    )
    grouped_by_model = pd.DataFrame(grouped_change_rows)
    grouped_ensemble = (
        grouped_by_model.groupby(
            [
                "scenario", "scenario_label", "period", "period_years",
                "grouping", "group", "statistic",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            baseline_value=("baseline_value", "first"),
            ensemble_future_value=("future_value", "mean"),
            change_percent=("change_percent", "mean"),
            inter_model_sd_change_percent=("change_percent", "std"),
            model_count=("model", "nunique"),
        )
    )
    multiscale_by_model = pd.DataFrame(multiscale_change_rows)
    multiscale_ensemble = (
        multiscale_by_model.groupby(
            [
                "scenario", "scenario_label", "period", "period_years",
                "time_scale", "statistic",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            baseline_value=("baseline_value", "first"),
            ensemble_future_value=("future_value", "mean"),
            change_percent=("change_percent", "mean"),
            inter_model_sd_change_percent=("change_percent", "std"),
            model_count=("model", "nunique"),
        )
    )

    float_format = f"%.{args.decimals}f"
    absolute.to_csv(output_dir / "01_absolute_statistics_by_model.csv", index=False, float_format=float_format)
    by_model.to_csv(output_dir / "02_percentage_change_by_model.csv", index=False, float_format=float_format)
    ensemble.to_csv(output_dir / "03_ensemble_percentage_change.csv", index=False, float_format=float_format)
    grouped_by_model.to_csv(
        output_dir / "04_monthly_seasonal_change_by_model.csv",
        index=False,
        float_format=float_format,
    )
    grouped_ensemble.to_csv(
        output_dir / "05_monthly_seasonal_ensemble_change.csv",
        index=False,
        float_format=float_format,
    )
    multiscale_by_model.to_csv(
        output_dir / "06_multiscale_change_by_model.csv",
        index=False,
        float_format=float_format,
    )
    multiscale_ensemble.to_csv(
        output_dir / "07_multiscale_ensemble_change.csv",
        index=False,
        float_format=float_format,
    )

    for period in EXPECTED_PERIODS:
        table = make_article_style_table(ensemble, period)
        try:
            table.to_csv(
                output_dir / f"table_ET_change_{period}.csv",
                encoding="utf-8-sig",
                float_format=float_format,
            )
        except PermissionError:
            print(f"Warning: skipped open/locked table_ET_change_{period}.csv")

        multiscale_table = make_multiscale_article_table(multiscale_ensemble, period)
        multiscale_table.to_csv(
            output_dir / f"table_ET_change_multiscale_{period}.csv",
            encoding="utf-8-sig",
            float_format=float_format,
        )

        for grouping in ("season", "month"):
            grouped_table = make_grouped_article_table(grouped_ensemble, period, grouping)
            grouped_name = f"table_ET_change_{grouping}_{period}.csv"
            try:
                grouped_table.to_csv(
                    output_dir / grouped_name,
                    encoding="utf-8-sig",
                    float_format=float_format,
                )
            except PermissionError:
                print(f"Warning: skipped open/locked {grouped_name}")

    baseline_out = pd.DataFrame(
        [{"statistic": k, "CWatM_1995_2019": v} for k, v in baseline_stats.items()]
    )
    baseline_out.to_csv(
        output_dir / "00_CWatM_baseline_statistics.csv", index=False, float_format=float_format
    )

    print(f"Validated 10 source files and wrote results to: {output_dir}")
    print(ensemble.to_string(index=False, float_format=lambda x: f"{x:.{args.decimals}f}"))


if __name__ == "__main__":
    main()
